In [33]:
from nrem_analysis.constant import RAW_DIR, PROCESSED_DIR, MOUSE_IDS_DUAL
from nrem_analysis.utils import plot_intervals

from replay_trajectory_classification import (
    SortedSpikesClassifier,
    Environment,
    RandomWalk,
    Uniform,
    Identity,
    DiagonalDiscrete,
    make_track_graph,
)

import pynapple as nap
import numpy as np
import matplotlib.pyplot as plt

data_dir = RAW_DIR / "dual"
mouse_id = "99b"

STATE_PROB = 0.99
STATE_NAMES = ["continuous", "fragmented", "stationary"]
BIN_SIZE_S = 0.001
DECODING_WINDOW = 500

session = nap.load_file(PROCESSED_DIR / "dual" / mouse_id / "session.npz")
hd_units = nap.load_file(PROCESSED_DIR / "dual" / mouse_id / "hd_units.npz")
turn_units = nap.load_file(PROCESSED_DIR / "dual" / mouse_id / "turn_units.npz")
head_direction = nap.load_file(PROCESSED_DIR / "dual" / mouse_id / "head_direction.npz")

### setup

In [ ]:
def get_environment(num_nodes: int = 360, place_bin_size: float = 1.0):
    radius = 180 / np.pi
    angle = np.linspace(2 * np.pi, 0, num=num_nodes, endpoint=False)
    node_positions = np.stack((radius * np.cos(angle), radius * np.sin(angle)), axis=1)

    node_ids = np.arange(node_positions.shape[0])
    edges = np.stack((node_ids, np.roll(node_ids, shift=1)), axis=1)

    track_graph = make_track_graph(node_positions, edges)

    n_nodes = len(track_graph.nodes)
    edge_order = np.stack(
        (np.roll(np.arange(n_nodes - 1, -1, -1), 1),
         np.arange(n_nodes - 1, -1, -1)),
        axis=1,
    )

    return Environment(
        place_bin_size=place_bin_size,
        track_graph=track_graph,
        edge_order=edge_order,
        edge_spacing=0,
    )

def fit_classifier(
    hd_spikes: nap.Tsd,
    hd_angle: nap.Tsd,
    train_ep: nap.IntervalSet,
    bin_size_ms: int = 1,
    place_bin_size: float = 1.0,
) -> SortedSpikesClassifier:
    """Fit the classifier on wake training data and return it."""
    spikes = (hd_spikes.count(bin_size=bin_size_ms, ep=train_ep, time_units="ms").astype(np.bool_))
    angle = circ_bin_average(tsd=hd_angle, bin_size=bin_size_ms, ep=train_ep, time_units="ms").to_numpy()

    # Build classifier
    environment = get_environment(place_bin_size=place_bin_size)
    continuous_transition_types = [
        [RandomWalk(movement_var=2.0), Uniform(), Identity()],
        [Uniform(),                    Uniform(), Uniform()],
        [RandomWalk(movement_var=2.0), Uniform(), Identity()],
    ]
    classifier = SortedSpikesClassifier(
        environments=environment,
        continuous_transition_types=continuous_transition_types,
        discrete_transition_type=DiagonalDiscrete(STATE_PROB),
    )
    
    # Fit classifier
    classifier.fit(angle, spikes.to_numpy())
    return classifier


### Decoding

In [77]:
nrem = sleep[sleep['state'] == 'nrem'][0]
ep = head_direction.time_support
ep = nap.IntervalSet(ep.start-BIN_SIZE_S/2, ep.end)
train_data = hd_units.count(bin_size=BIN_SIZE_S, ep=ep, time_units="s").astype(np.bool_)

assert train_data.shape[0] == head_direction.shape[0]

In [80]:
environment = get_environment(place_bin_size=1.0)
continuous_transition_types = [
    [RandomWalk(movement_var=2.0), Uniform(), Identity()],
    [Uniform(),                    Uniform(), Uniform()],
    [RandomWalk(movement_var=2.0), Uniform(), Identity()],
]

classifier = SortedSpikesClassifier(
    environments=environment,
    continuous_transition_types = continuous_transition_types,
    discrete_transition_type=DiagonalDiscrete(STATE_PROB),
)

In [81]:
classifier.fit(head_direction, train_data)

ValueError: assignment destination is read-only